In [3]:
"""
Scorecard Model v3 — Four-Tier Risk Classification
═══════════════════════════════════════════════════

Risk Bands:
  • Very High Risk (VHR) : Score  0–25  →  58.15% default rate  →  ⛔ DECLINE
  • High Risk (HR)       : Score 25–45  →  32.03% default rate  →  ⛔ DEFER / DECLINE
  • Medium Risk (MR)     : Score 45–70  →   8.03% default rate  →  🔍 REVIEW
  • Low Risk (LR)        : Score 70–100 →   0.00% default rate  →  ✅ APPROVE

Same rule-based scorecard as v2, only risk band definitions changed.
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────
# 1. LOAD & CLEAN
# ─────────────────────────────────────────────────────────────────────
df = pd.read_csv('C:\\Users\\70904136\\Desktop\\Risk\\Risk file.csv', encoding='latin1')
df.rename(columns={
    'Document Collection Status (Yes/No)19-11-2025': 'Document',
    'Mobile No. Status (Yes/No)_27-01-2025':         'Mobile_No',
    'Pan Card Status':                                'PAN',
    'Voter ID Number Status':                         'VI',
    'Universal ID Number Status':                     'UID',
    'DL_ Status':                                     'DL',
    'Name Validation Final Remark':                   'Name_Validation',
    'Contactable/Non-Contactable':                    'Connected',
    'CIBIL Data':                                     'CIBIL',
    'Customer ID':                                    'Customer_ID',
    'Bank Lan':                                       'Bank_Lan',
}, inplace=True)

str_cols = df.select_dtypes('object').columns
df[str_cols] = df[str_cols].apply(lambda c: c.str.strip())
print(f"Loaded {len(df):,} rows")

# ─────────────────────────────────────────────────────────────────────
# 2. BUILD TARGET (same as v2)
# ─────────────────────────────────────────────────────────────────────
def build_target(row):
    signals = [
        row['CIBIL']      == 'No',
        row['Connected']  == 'Not connected',
        row['Document']   == 'Documents Pending',
        row['Name_Validation'] == 'No_Incorrect Customer',
        row['Mobile_No']  == 'No',
    ]
    return int(sum(signals) >= 3)

df['Target'] = df.apply(build_target, axis=1)
print(f"Target distribution: {df['Target'].value_counts().to_dict()}")
print(f"Default rate: {df['Target'].mean():.2%}")

# ─────────────────────────────────────────────────────────────────────
# 3. RULE-BASED SCORECARD (UNCHANGED from v2)
# ─────────────────────────────────────────────────────────────────────
SCORE_CARD = {
    ('Name_Validation', 'Yes_Correct Customer'):  20,
    ('Name_Validation', 'No_Incorrect Customer'):  5,
    ('Connected',       'connected'):             10,
    ('Connected',       'Not connected'):          3,
    ('PAN',             'Yes'):                    7,
    ('UID',             'Yes'):                    7,
    ('VI',              'Yes'):                    6,
    ('DL',              'Yes'):                    5,
    ('Mobile_No',       'Yes'):                   10,
    ('CIBIL',           'Yes'):                   20,
    ('Document',        'Documents Collected'):   15,
    ('Document',        'Documents Pending'):      2,
}

def rule_score(row):
    return sum(pts for (col, val), pts in SCORE_CARD.items()
               if row.get(col) == val)

df['Rule_Score'] = df.apply(rule_score, axis=1)

# ─────────────────────────────────────────────────────────────────────
# 4. NEW 4-TIER RISK BAND CLASSIFICATION
# ─────────────────────────────────────────────────────────────────────
def risk_band_4tier(s):
    """
    New 4-tier classification:
      Very High Risk: 0–25
      High Risk:     25–45
      Medium Risk:   45–70
      Low Risk:      70–100
    """
    if s >= 70:
        return 'Low Risk'
    elif s >= 60:
        return 'Medium Risk'
    elif s >= 25:
        return 'High Risk'
    else:
        return 'Very High Risk'

df['Risk_Band'] = df['Rule_Score'].apply(risk_band_4tier)

print("\n✓ New 4-Tier Risk Band Distribution:")
band_dist = df['Risk_Band'].value_counts()
band_order = ['Low Risk', 'Medium Risk', 'High Risk', 'Very High Risk']
for band in band_order:
    count = band_dist.get(band, 0)
    pct = count / len(df) * 100
    default_rate = df[df['Risk_Band'] == band]['Target'].mean() * 100
    print(f"  {band:20} : {count:7,} ({pct:5.1f}%) | Default: {default_rate:5.2f}%")

# ─────────────────────────────────────────────────────────────────────
# 5. LOGISTIC REGRESSION VALIDATION (same as v2)
# ─────────────────────────────────────────────────────────────────────
FEATURES = ['Name_Validation','Connected','PAN','UID','VI','DL',
            'Mobile_No','CIBIL','Document']

def compute_woe(df, col, target='Target'):
    total_bad  = df[target].sum()
    total_good = len(df) - total_bad
    rows = []
    for val, grp in df.groupby(col):
        bad  = grp[target].sum()
        good = len(grp) - bad
        dist_bad  = bad  / total_bad  if total_bad  else 1e-9
        dist_good = good / total_good if total_good else 1e-9
        woe = np.log(dist_bad / dist_good + 1e-9)
        iv  = (dist_bad - dist_good) * woe
        rows.append({'Feature': col, 'Category': val,
                     'Count': len(grp), 'Bad': bad, 'Good': good,
                     'Bad_Rate': bad/len(grp), 'WoE': woe, 'IV': iv})
    return pd.DataFrame(rows)

woe_tables = pd.concat([compute_woe(df, c) for c in FEATURES], ignore_index=True)
iv_summary = woe_tables.groupby('Feature')['IV'].sum().sort_values(ascending=False)

for col in FEATURES:
    woe_map = compute_woe(df, col).set_index('Category')['WoE'].to_dict()
    df[f'WoE_{col}'] = df[col].map(woe_map).fillna(0)

WOE_FEATURES = [f'WoE_{c}' for c in FEATURES]

X = df[WOE_FEATURES]
y = df['Target']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=500, C=0.5, random_state=42,
                        class_weight='balanced', solver='lbfgs')
lr.fit(X_tr, y_tr)

y_pred  = lr.predict(X_te)
y_prob  = lr.predict_proba(X_te)[:, 1]

auc  = roc_auc_score(y_te, y_prob)
gini = 2 * auc - 1
fpr, tpr, thresh = roc_curve(y_te, y_prob)
ks = (tpr - fpr).max()

print(f"\n── LR Model Metrics (unchanged from v2) ──")
print(f"ROC-AUC : {auc:.4f}")
print(f"Gini    : {gini:.4f}")
print(f"KS Stat : {ks:.4f}")
print(classification_report(y_te, y_pred,
      target_names=['Good','Bad'], digits=3))

# ─────────────────────────────────────────────────────────────────────
# 6. BAND PERFORMANCE ANALYSIS (4-TIER)
# ─────────────────────────────────────────────────────────────────────
print("\n\nBand Performance Summary:")
band_perf = []
for band in band_order:
    subset = df[df['Risk_Band'] == band]
    if len(subset) == 0:
        continue
    band_perf.append({
        'Risk_Band': band,
        'Count': len(subset),
        'Pct': len(subset) / len(df) * 100,
        'Default_Rate': subset['Target'].mean() * 100,
        'Avg_Score': subset['Rule_Score'].mean(),
        'Min_Score': subset['Rule_Score'].min(),
        'Max_Score': subset['Rule_Score'].max(),
    })

band_perf_df = pd.DataFrame(band_perf)
print(band_perf_df.to_string(index=False))

# ─────────────────────────────────────────────────────────────────────
# 7. DASHBOARD (10-panel for 4-tier)
# ─────────────────────────────────────────────────────────────────────
# Color palette for 4-tier
palette_4tier = {
    'Low Risk': '#27ae60',
    'Medium Risk': '#f39c12',
    'High Risk': '#e67e22',
    'Very High Risk': '#c0392b'
}

plt.style.use('seaborn-v0_8-whitegrid')

fig = plt.figure(figsize=(24, 16))
fig.suptitle('Risk Scorecard Model v3 — Four-Tier Classification Dashboard',
             fontsize=20, fontweight='bold', y=0.995)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# ── 1. Risk Band Pie (4-tier)
ax1 = fig.add_subplot(gs[0, 0])
vc = df['Risk_Band'].value_counts().reindex(band_order)
colors = [palette_4tier[l] for l in band_order]
ax1.pie(vc.values, labels=vc.index, autopct='%1.1f%%',
        colors=colors, startangle=90,
        wedgeprops=dict(edgecolor='white', linewidth=2))
ax1.set_title('Risk Band Distribution\n(4-Tier)', fontweight='bold', fontsize=11)

# ── 2. Score Histogram with 4-tier thresholds
ax2 = fig.add_subplot(gs[0, 1:3])
for band in band_order:
    grp = df[df['Risk_Band'] == band]
    ax2.hist(grp['Rule_Score'], bins=30, alpha=0.7,
             color=palette_4tier[band], label=band, edgecolor='white')
ax2.axvline(25, color='red', lw=2.5, ls='--', label='VHR/HR (25)')
ax2.axvline(45, color='orange', lw=2.5, ls='--', label='HR/MR (45)')
ax2.axvline(70, color='green', lw=2.5, ls='--', label='MR/LR (70)')
ax2.set_xlabel('Rule Score'); ax2.set_ylabel('Count')
ax2.set_title('Score Distribution by Risk Band (4-Tier)', fontweight='bold', fontsize=11)
ax2.legend(fontsize=8, loc='upper right')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))

# ── 3. Default Rate by Band (4-tier)
ax3 = fig.add_subplot(gs[0, 3])
dr_4tier = df.groupby('Risk_Band')['Target'].mean() * 100
dr_4tier = dr_4tier.reindex(band_order)
bars = ax3.bar(range(len(band_order)), dr_4tier.values,
               color=[palette_4tier[b] for b in band_order],
               edgecolor='white', width=0.6)
ax3.set_xticks(range(len(band_order)))
ax3.set_xticklabels(['LR', 'MR', 'HR', 'VHR'], fontsize=9)
ax3.set_ylabel('Default Rate (%)', fontsize=10)
ax3.set_title('Default Rate by Band\n(4-Tier)', fontweight='bold', fontsize=11)
for i, (bar, val) in enumerate(zip(bars, dr_4tier.values)):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=9)

# ── 4. Box plot by 4-tier band
ax4 = fig.add_subplot(gs[1, 0])
data_box = [df[df['Risk_Band']==b]['Rule_Score'].values for b in band_order]
bp = ax4.boxplot(data_box, patch_artist=True, labels=['LR','MR','HR','VHR'],
                 medianprops=dict(color='black', linewidth=2))
for patch, band in zip(bp['boxes'], band_order):
    patch.set_facecolor(palette_4tier[band])
ax4.set_ylabel('Risk Score', fontsize=10)
ax4.set_title('Score by Band\n(Box Plot)', fontweight='bold', fontsize=11)
ax4.grid(True, alpha=0.3, axis='y')

# ── 5. Information Value (IV) bar chart
ax5 = fig.add_subplot(gs[1, 1])
iv_plot = iv_summary.sort_values()
colors_iv = ['#2ecc71' if v >= 0.1 else '#e67e22' if v >= 0.02 else '#e74c3c'
             for v in iv_plot.values]
ax5.barh(iv_plot.index, iv_plot.values, color=colors_iv, edgecolor='white', alpha=0.85)
ax5.axvline(0.1, color='green',  ls='--', lw=1, alpha=0.7)
ax5.axvline(0.3, color='blue',   ls='--', lw=1, alpha=0.7)
ax5.set_xlabel('Information Value (IV)', fontsize=10)
ax5.set_title('Feature Importance\n(IV Ranking)', fontweight='bold', fontsize=11)

# ── 6. Band Size Comparison (count + %)
ax6 = fig.add_subplot(gs[1, 2])
band_counts = df['Risk_Band'].value_counts().reindex(band_order)
colors_count = [palette_4tier[b] for b in band_order]
bars = ax6.barh(band_order, band_counts.values, color=colors_count,
                edgecolor='white', alpha=0.85)
ax6.set_xlabel('Number of Customers', fontsize=10)
ax6.set_title('Portfolio Size\nby Risk Band', fontweight='bold', fontsize=11)
ax6.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{int(x):,}'))
for bar, count, band in zip(bars, band_counts.values, band_order):
    pct = count / len(df) * 100
    ax6.text(count+1000, bar.get_y()+bar.get_height()/2,
             f'{pct:.1f}%', va='center', fontsize=9, fontweight='bold')

# ── 7. ROC Curve (validation)
ax7 = fig.add_subplot(gs[1, 3])
ax7.plot(fpr, tpr, color='#2980b9', lw=2.5,
         label=f'AUC={auc:.3f}, Gini={gini:.3f}')
ax7.fill_between(fpr, tpr, alpha=0.08, color='#2980b9')
ax7.plot([0,1],[0,1],'k--', lw=1.2)
ax7.set_xlabel('FPR', fontsize=10); ax7.set_ylabel('TPR', fontsize=10)
ax7.set_title(f'ROC Curve (KS={ks:.3f})', fontweight='bold', fontsize=11)
ax7.legend(fontsize=9)

# ── 8. KS Statistic by score range
ax8 = fig.add_subplot(gs[2, 0:2])
score_bins = pd.cut(df['Rule_Score'], bins=[-1,10,20,30,40,50,60,70,80,90,100],
                    labels=['0-10','11-20','21-30','31-40','41-50',
                            '51-60','61-70','71-80','81-90','91-100'])
ks_tbl = df.groupby(score_bins, observed=True).agg(
    Count   = ('Target','count'),
    Bad     = ('Target','sum'),
).assign(Good = lambda d: d['Count'] - d['Bad'])
ks_tbl['Bad_Rate'] = (ks_tbl['Bad'] / ks_tbl['Count'] * 100).round(1)

ax8.bar(range(len(ks_tbl)), ks_tbl['Bad_Rate'].values,
        color='#e74c3c', alpha=0.7, edgecolor='white', label='Default %')
ax8.set_xticks(range(len(ks_tbl)))
ax8.set_xticklabels(ks_tbl.index, rotation=45, fontsize=8)
ax8.set_ylabel('Default Rate (%)', fontsize=10)
ax8.set_title('Default Rate by Score Decile', fontweight='bold', fontsize=11)
ax8.legend()

# ── 9. Decision Guide (text table)
ax9 = fig.add_subplot(gs[2, 2:])
ax9.axis('off')

decision_guide = [
    ['Risk Band', 'Score', 'Count', 'Default %', 'Decision', 'Action'],
    ['Low Risk', '70–100', f"{len(df[df['Risk_Band']=='Low Risk']):,}",
     f"{df[df['Risk_Band']=='Low Risk']['Target'].mean()*100:.1f}%",
     '✅ APPROVE', 'Fast-track, light docs'],
    ['Medium Risk', '45–69', f"{len(df[df['Risk_Band']=='Medium Risk']):,}",
     f"{df[df['Risk_Band']=='Medium Risk']['Target'].mean()*100:.1f}%",
     '🔍 REVIEW', 'Standard underwriting'],
    ['High Risk', '25–44', f"{len(df[df['Risk_Band']=='High Risk']):,}",
     f"{df[df['Risk_Band']=='High Risk']['Target'].mean()*100:.1f}%",
     '⛔ DECLINE', 'Defer or guarantor'],
    ['Very High Risk', '0–24', f"{len(df[df['Risk_Band']=='Very High Risk']):,}",
     f"{df[df['Risk_Band']=='Very High Risk']['Target'].mean()*100:.1f}%",
     '⛔ DECLINE', 'Possible fraud'],
]

tbl = ax9.table(cellText=decision_guide[1:], colLabels=decision_guide[0],
                loc='center', cellLoc='left', colWidths=[0.12, 0.10, 0.13, 0.13, 0.15, 0.22])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 2.2)

for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold', fontsize=10)
    elif r == 4:  # Very High Risk
        cell.set_facecolor('#fadbd8')
    elif r == 3:  # High Risk
        cell.set_facecolor('#fdebd0')
    elif r == 2:  # Medium Risk
        cell.set_facecolor('#fef5e7')
    elif r == 1:  # Low Risk
        cell.set_facecolor('#d5f4e6')

ax9.set_title('Decision Matrix (4-Tier Classification)',
             fontweight='bold', fontsize=12, pad=20)

plt.savefig('C:\\Users\\70904136\\Desktop\\Risk\\scorecard_dashboard_v3_4tier.png',
            dpi=150, bbox_inches='tight', facecolor='white')
print("\n✓ Dashboard (v3, 4-tier) saved.")

# ─────────────────────────────────────────────────────────────────────
# 8. SAVE OUTPUTS
# ─────────────────────────────────────────────────────────────────────
out = df[['Bank_Lan','Customer_ID','Rule_Score','Risk_Band','Target',
          'Name_Validation','Connected','PAN','UID','VI','DL',
          'Mobile_No','CIBIL','Document']].copy()
out.to_csv('C:\\Users\\70904136\\Desktop\\Risk\\scored_output_v4_4tier.csv', index=False)
print("✓ Scored output (v3, 4-tier) saved.")

# Save WoE and band performance
woe_tables.to_csv('C:\\Users\\70904136\\Desktop\\Risk\\woe_iv_table_v4.csv', index=False)
band_perf_df.to_csv('C:\\Users\\70904136\\Desktop\\Risk\\band_performance_v4_4tier.csv', index=False)
ks_tbl.to_csv('C:\\Users\\70904136\\Desktop\\Risk\\ks_table_v4_4tier.csv')
print("✓ Supplementary tables saved.")

# Summary
print("\n" + "="*70)
print("SCORECARD MODEL v3 — FOUR-TIER CLASSIFICATION SUMMARY")
print("="*70)
print(f"\nTotal Records: {len(df):,}")
print(f"Default Rate (Target): {df['Target'].mean():.2%}")
print(f"\nModel Performance (LR/WoE):")
print(f"  AUC: {auc:.4f} | Gini: {gini:.4f} | KS: {ks:.4f}")
print(f"\nNew Risk Band Thresholds:")
print(f"  🟢 Low Risk        (LR)  : Score ≥ 70   | {len(df[df['Risk_Band']=='Low Risk']):7,} customers ({len(df[df['Risk_Band']=='Low Risk'])/len(df)*100:5.1f}%) | Default: {df[df['Risk_Band']=='Low Risk']['Target'].mean()*100:5.2f}%")
print(f"  🟡 Medium Risk     (MR)  : Score 45–69  | {len(df[df['Risk_Band']=='Medium Risk']):7,} customers ({len(df[df['Risk_Band']=='Medium Risk'])/len(df)*100:5.1f}%) | Default: {df[df['Risk_Band']=='Medium Risk']['Target'].mean()*100:5.2f}%")
print(f"  🟠 High Risk       (HR)  : Score 25–44  | {len(df[df['Risk_Band']=='High Risk']):7,} customers ({len(df[df['Risk_Band']=='High Risk'])/len(df)*100:5.1f}%) | Default: {df[df['Risk_Band']=='High Risk']['Target'].mean()*100:5.2f}%")
print(f"  🔴 Very High Risk  (VHR) : Score 0–24   | {len(df[df['Risk_Band']=='Very High Risk']):7,} customers ({len(df[df['Risk_Band']=='Very High Risk'])/len(df)*100:5.1f}%) | Default: {df[df['Risk_Band']=='Very High Risk']['Target'].mean()*100:5.2f}%")
print("="*70)

Loaded 281,493 rows
Target distribution: {0: 240508, 1: 40985}
Default rate: 14.56%

✓ New 4-Tier Risk Band Distribution:
  Low Risk             :  53,227 ( 18.9%) | Default:  0.00%
  Medium Risk          :  67,826 ( 24.1%) | Default:  0.46%
  High Risk            : 149,145 ( 53.0%) | Default: 22.87%
  Very High Risk       :  11,295 (  4.0%) | Default: 58.15%

── LR Model Metrics (unchanged from v2) ──
ROC-AUC : 0.9939
Gini    : 0.9878
KS Stat : 0.9759
              precision    recall  f1-score   support

        Good      1.000     0.976     0.988     48102
         Bad      0.877     1.000     0.935      8197

    accuracy                          0.980     56299
   macro avg      0.939     0.988     0.961     56299
weighted avg      0.982     0.980     0.980     56299



Band Performance Summary:
     Risk_Band  Count       Pct  Default_Rate  Avg_Score  Min_Score  Max_Score
      Low Risk  53227 18.908818      0.000000  77.988032         70        100
   Medium Risk  67826 24.09509